# NAMIBIA UNIVERSITY OF SCIENCE AND TECHNOLOGY
# MASTER OF DATA SCIENCE
# TRENDS IN ARTIFICIAL INTELLIGENCE AND MACHINE LEARNING (TAI911S)
## ASSIGNMENT 2

### Done By:
#### Name and Student Number: Sakaria Nekwaya 214059286
#### Name and Student Number: Hambeleleni Shaningwa 213091704

## PROBLEM 2

#### Load Required Packages

In [257]:
using ARFFFiles, DataFrames, MLJ, MLJBase, MLJModels, MLJScikitLearnInterface, DecisionTree, MLJLinearModels, Plots, ROCAnalysis, Random

#### Loading the ARFF dataset and processing it

In [259]:
beans_data = ARFFFiles.load("ag_soybean.arff.txt") #Load the dataset
beans_df = DataFrame(beans_data) #Convert to dataframe
beans_df = coalesce.(beans_df, missing) #Handle missing values
beans_df = dropmissing(beans_df, :class) #Remove rows with missing target values

Row,date,plant-stand,precip,temp,hail,crop-hist,area-damaged,severity,seed-tmt,germination,plant-growth,leaves,leafspots-halo,leafspots-marg,leafspot-size,leaf-shread,leaf-malf,leaf-mild,stem,lodging,stem-cankers,canker-lesion,fruiting-bodies,external-decay,mycelium,int-discolor,sclerotia,fruit-pods,fruit-spots,seed,mold-growth,seed-discolor,seed-size,shriveling,roots,class
,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…
1,october,normal,gt-norm,norm,yes,same-lst-yr,low-areas,pot-severe,none,90-100,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,no,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
2,august,normal,gt-norm,norm,yes,same-lst-two-yrs,scattered,severe,fungicide,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
3,july,normal,gt-norm,norm,yes,same-lst-yr,scattered,severe,fungicide,lt-80,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,dna,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
4,july,normal,gt-norm,norm,yes,same-lst-yr,scattered,severe,none,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,dna,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
5,october,normal,gt-norm,norm,yes,same-lst-two-yrs,scattered,pot-severe,none,lt-80,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
6,september,normal,gt-norm,norm,yes,same-lst-sev-yrs,scattered,pot-severe,none,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,dna,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
7,september,normal,gt-norm,norm,yes,same-lst-two-yrs,scattered,pot-severe,fungicide,90-100,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,no,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
8,august,normal,gt-norm,norm,no,same-lst-yr,scattered,pot-severe,none,lt-80,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker
9,october,normal,gt-norm,norm,yes,same-lst-sev-yrs,scattered,pot-severe,fungicide,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm,diaporthe-stem-canker


#### Convert data features to appropriate data types

In [263]:
coerce!(beans_df, Count => Continuous)
for col in names(beans_df)
    if eltype(beans_df[!, col]) <: AbstractString
        coerce!(beans_df, col => Multiclass)
    end
end

#### Split the data into features (X) and target (y)

In [231]:
y  = beans_df.class
X = DataFrames.select(beans_df, Not(:class))

Row,date,plant-stand,precip,temp,hail,crop-hist,area-damaged,severity,seed-tmt,germination,plant-growth,leaves,leafspots-halo,leafspots-marg,leafspot-size,leaf-shread,leaf-malf,leaf-mild,stem,lodging,stem-cankers,canker-lesion,fruiting-bodies,external-decay,mycelium,int-discolor,sclerotia,fruit-pods,fruit-spots,seed,mold-growth,seed-discolor,seed-size,shriveling,roots
,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?,Cat…?
1,october,normal,gt-norm,norm,yes,same-lst-yr,low-areas,pot-severe,none,90-100,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,no,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
2,august,normal,gt-norm,norm,yes,same-lst-two-yrs,scattered,severe,fungicide,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
3,july,normal,gt-norm,norm,yes,same-lst-yr,scattered,severe,fungicide,lt-80,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,dna,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
4,july,normal,gt-norm,norm,yes,same-lst-yr,scattered,severe,none,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,dna,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
5,october,normal,gt-norm,norm,yes,same-lst-two-yrs,scattered,pot-severe,none,lt-80,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
6,september,normal,gt-norm,norm,yes,same-lst-sev-yrs,scattered,pot-severe,none,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,dna,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
7,september,normal,gt-norm,norm,yes,same-lst-two-yrs,scattered,pot-severe,fungicide,90-100,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,no,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
8,august,normal,gt-norm,norm,no,same-lst-yr,scattered,pot-severe,none,lt-80,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm
9,october,normal,gt-norm,norm,yes,same-lst-sev-yrs,scattered,pot-severe,fungicide,80-89,abnorm,abnorm,absent,dna,dna,absent,absent,absent,abnorm,yes,above-sec-nde,brown,present,firm-and-dry,absent,none,absent,norm,dna,norm,absent,absent,norm,absent,norm


In [233]:
schema(X) #confirm features type

┌─────────────────┬───────────────────────────────┬─────────────────────────────
│ names           │ scitypes                      │ types                      ⋯
├─────────────────┼───────────────────────────────┼─────────────────────────────
│ date            │ Union{Missing, Multiclass{7}} │ Union{Missing, Categorical ⋯
│ plant-stand     │ Union{Missing, Multiclass{2}} │ Union{Missing, Categorical ⋯
│ precip          │ Union{Missing, Multiclass{3}} │ Union{Missing, Categorical ⋯
│ temp            │ Union{Missing, Multiclass{3}} │ Union{Missing, Categorical ⋯
│ hail            │ Union{Missing, Multiclass{2}} │ Union{Missing, Categorical ⋯
│ crop-hist       │ Union{Missing, Multiclass{4}} │ Union{Missing, Categorical ⋯
│ area-damaged    │ Union{Missing, Multiclass{4}} │ Union{Missing, Categorical ⋯
│ severity        │ Union{Missing, Multiclass{3}} │ Union{Missing, Categorical ⋯
│ seed-tmt        │ Union{Missing, Multiclass{3}} │ Union{Missing, Categorical ⋯
│ germination     │ Union{Mi

#### Split the data into 70% training data and 30% testing data

In [265]:
Random.seed!(123)
train, test = partition(eachindex(y), 0.7, shuffle=true)

([226, 68, 402, 3, 580, 192, 360, 24, 561, 266  …  147, 183, 663, 215, 428, 272, 448, 189, 369, 95], [441, 411, 83, 596, 566, 207, 382, 349, 317, 143  …  358, 135, 231, 429, 410, 64, 211, 407, 308, 62])

### Define the models: Define Decision Tree, Boosting (EvoTree), Bagging (Random Forest), and Stacking.

In [267]:
using MLJDecisionTreeInterface

#### Decision Tree Model

In [271]:
DecisionTreeClassifier = @load DecisionTreeClassifier pkg=DecisionTree verbosity=0
tree_model = DecisionTreeClassifier()

DecisionTreeClassifier(
  max_depth = -1, 
  min_samples_leaf = 1, 
  min_samples_split = 2, 
  min_purity_increase = 0.0, 
  n_subfeatures = 0, 
  post_prune = false, 
  merge_purity_threshold = 1.0, 
  display_depth = 5, 
  feature_importance = :impurity, 
  rng = TaskLocalRNG())

In [269]:
using MLJScikitLearnInterface
using MLJ

#### Boosting Method

In [273]:
EvoTreeClassifier = @load EvoTreeClassifier verbosity=0
evo_model = EvoTreeClassifier()

EvoTreeClassifier(
  loss = :mlogloss, 
  metric = :mlogloss, 
  nrounds = 100, 
  early_stopping_rounds = 9223372036854775807, 
  L2 = 1.0, 
  lambda = 0.0, 
  gamma = 0.0, 
  eta = 0.1, 
  max_depth = 6, 
  min_weight = 1.0, 
  rowsample = 1.0, 
  colsample = 1.0, 
  nbins = 64, 
  alpha = 0.5, 
  tree_type = :binary, 
  rng = MersenneTwister(123), 
  device = :cpu)

#### Bagging Method

In [275]:
RandomForestClassifier = @load RandomForestClassifier pkg=DecisionTree verbosity=0
forest_model = RandomForestClassifier()

RandomForestClassifier(
  max_depth = -1, 
  min_samples_leaf = 1, 
  min_samples_split = 2, 
  min_purity_increase = 0.0, 
  n_subfeatures = -1, 
  n_trees = 100, 
  sampling_fraction = 0.7, 
  feature_importance = :impurity, 
  rng = TaskLocalRNG())

#### Stacking Method

In [215]:

LogisticClassifier = @load LogisticClassifier pkg=MLJLinearModels verbosity=0
Logistic_model = LogisticClassifier()
stack_model = Stack(metalearner=Logistic_model, model1=tree_model, model2=forest_model, model3=evo_model)

ProbabilisticStack(
  metalearner = LogisticClassifier(
        lambda = 2.220446049250313e-16, 
        gamma = 0.0, 
        penalty = :l2, 
        fit_intercept = true, 
        penalize_intercept = false, 
        scale_penalty_with_samples = true, 
        solver = nothing), 
  resampling = CV(
        nfolds = 6, 
        shuffle = false, 
        rng = TaskLocalRNG()), 
  measures = nothing, 
  cache = true, 
  acceleration = CPU1{Nothing}(nothing), 
  model1 = DecisionTreeClassifier(
        max_depth = -1, 
        min_samples_leaf = 1, 
        min_samples_split = 2, 
        min_purity_increase = 0.0, 
        n_subfeatures = 0, 
        post_prune = false, 
        merge_purity_threshold = 1.0, 
        display_depth = 5, 
        feature_importance = :impurity, 
        rng = TaskLocalRNG()), 
  model2 = RandomForestClassifier(
        max_depth = -1, 
        min_samples_leaf = 1, 
        min_samples_split = 2, 
        min_purity_increase = 0.0, 
        n_subfeatures 

### Evaluate Models
#### Train and evaluate all models

In [235]:
function evaluate_model(model, X_train, y_train, X_test, y_test)
    mach = machine(model, X_train, y_train)
    fit!(mach)
    y_pred = predict_mode(mach, X_test)
    y_probs = predict(mach, X_test)
    
    # Calculate metrics
    acc = accuracy(y_pred, y_test)
    prec = precision(y_pred, y_test, average=:macro)
    rec = recall(y_pred, y_test, average=:macro)
    
    # Calculate AUC (One-vs-Rest)
    levels_ = levels(y_train)
    auc_scores = Float64[]
    for class in levels_
        y_true_binary = y_test .== class
        y_score = pdf.(y_probs, class)
        roc = ROCAnalysis.roc(y_true_binary, y_score)
        push!(auc_scores, ROCAnalysis.auc(roc))
    end
    auc = mean(auc_scores)
    
    return (acc, prec, rec, auc)
end

evaluate_model (generic function with 1 method)

#### Evaluate the models

In [ ]:
models = [
    ("Decision Tree", tree_model),
    ("Boosting", evo_model),
    ("Bagging", forest_model),
    ("Stacking", stack_model)
]

results = []
for (name, model) in models
    acc, prec, rec, auc = evaluate_model(model, X[train,:], y[train], X[test,:], y[test])
    push!(results, (name, acc, prec, rec, auc))
end

#### Form a dataframe of results

In [239]:
using DataFrames
result_df = DataFrame(
    Model = [r[1] for r in results],
    Accuracy = [r[2] for r in results],
    Precision = [r[3] for r in results],
    Recall = [r[4] for r in results],
    AUC = [r[5] for r in results]
)

Row,Model,Accuracy,Precision,Recall,AUC
,Any,Any,Any,Any,Any


#### Plot ROC for Stacking

In [ ]:

mach_stacking = machine(stack_model, X[train,:], y[train])
fit!(mach_stacking)
y_probs_stacking = predict(mach_stacking, X[test,:])

levels_stacking = levels(y)
stacking_plot = plot(title="ROC Curves (Stacking)", xlabel="False Positive Rate", ylabel="True Positive Rate")

for (i, class) in enumerate(levels_stacking)
    y_true_binary = y[test] .== class
    y_score = pdf.(y_probs_stacking, class)
    roc = ROCAnalysis.roc(y_true_binary, y_score)
    plot!(roc, label="Class $class")
end
display(stacking_plot)

#### Analysis

##### Stacking achieved the highest performance, demonstrating the power of combining diverse base learners.
##### Random Forest outperformed EvoTree, likely due to its inherent robustness against overfitting